In [1]:
#cell 1
# Check GPU.
!nvidia-smi

Tue Jun  2 13:25:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
#cell 2
# Install dependencies.
!pip -q install -U uv

# Basic dependencies.
!uv pip install --system -U openai tqdm requests psutil pandas

# Install recent vLLM nightly for CUDA 13.0 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

# Version check.
import sys
import torch
import vllm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)

Using Python 3.12.13 environment at: /usr
Resolved 24 packages in 75ms
Prepared 1 package in 0.22ms
Uninstalled 1 package in 8ms
Installed 1 package in 11ms
 - numpy==2.3.5
 + numpy==2.4.6
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 6.77s
Prepared 1 package in 0.34ms
Uninstalled 1 package in 9ms
Installed 1 package in 11ms
 - numpy==2.4.6
 + numpy==2.3.5
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
vLLM: 0.22.1rc1.dev70+g0eeba5eec


In [3]:
#cell 3
# Mount Google Drive and prepare paths.
from google.colab import drive
from pathlib import Path
import shutil
import json
import os

drive.mount("/content/drive")

GDRIVE_INPUT_DIR = Path("/content/drive/MyDrive/final_project/idea_1/answers/hotpotqa")

INPUT_FILES = [
    "hotpotqa_answer_qwen3.5.json",
    "hotpotqa_answer_gpt_oss.json",
    "hotpotqa_answer_gemma4.json",
]

LOCAL_WORK_DIR = Path("/content/hotpotqa_llm_judge")
LOCAL_INPUT_DIR = LOCAL_WORK_DIR / "inputs"
LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

GDRIVE_OUTPUT_DIR = GDRIVE_INPUT_DIR / "llm_judge_qwen35_27b"
GDRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Copy files to local disk.
for file_name in INPUT_FILES:
    src = GDRIVE_INPUT_DIR / file_name
    dst = LOCAL_INPUT_DIR / file_name

    assert src.exists(), f"Input file not found: {src}"

    shutil.copy2(src, dst)
    print("Copied:", src, "->", dst)

print("Local input dir:", LOCAL_INPUT_DIR)
print("Output dir:", GDRIVE_OUTPUT_DIR)

Mounted at /content/drive
Copied: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_qwen3.5.json -> /content/hotpotqa_llm_judge/inputs/hotpotqa_answer_qwen3.5.json
Copied: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gpt_oss.json -> /content/hotpotqa_llm_judge/inputs/hotpotqa_answer_gpt_oss.json
Copied: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gemma4.json -> /content/hotpotqa_llm_judge/inputs/hotpotqa_answer_gemma4.json
Local input dir: /content/hotpotqa_llm_judge/inputs
Output dir: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b


In [4]:
#cell 4
# Load input JSON files.
from collections import Counter

def load_json_list(path: Path):
    # Load a JSON list.
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    assert isinstance(data, list), f"Expected a list in {path}"
    return data

all_input_data = {}

for file_name in INPUT_FILES:
    path = LOCAL_INPUT_DIR / file_name
    data = load_json_list(path)
    all_input_data[file_name] = data

    print("\nFile:", file_name)
    print("Rows:", len(data))
    print("Types:", Counter(x.get("type") for x in data))

    if len(data) > 0:
        print("Keys:", sorted(data[0].keys()))


File: hotpotqa_answer_qwen3.5.json
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['attempt', 'completion_tokens', 'gt', 'max_tokens_used', 'question', 'response', 'retry_reason', 'source_index', 'type']

File: hotpotqa_answer_gpt_oss.json
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['attempt', 'completion_tokens', 'gt', 'max_tokens_used', 'question', 'reasoning_tokens', 'response', 'retry_reason', 'source_index', 'type']

File: hotpotqa_answer_gemma4.json
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['attempt', 'completion_tokens', 'gt', 'max_tokens_used', 'question', 'response', 'retry_reason', 'source_index', 'type']


In [5]:
#cell 5
# Start vLLM server in Qwen non-thinking mode.
import subprocess
import time
import requests
import shlex
import psutil
from pathlib import Path
import os

MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Total context length.
MAX_MODEL_LEN = 12288

# Memory margin for RTX PRO 6000 Blackwell 96GB.
GPU_MEMORY_UTILIZATION = 0.92

# Throughput settings.
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))

        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass

        parent.kill()
        parent.wait(timeout=10)
        print("Killed old process tree:", pid)

    except Exception:
        pass

# Stop old PID.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Kill leftover vLLM servers.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode.
    "--language-model-only",

    # Qwen non-thinking mode.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    # Throughput settings.
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    # Shared prompt prefix cache.
    "--enable-prefix-caching",

    # Use vLLM generation config.
    "--generation-config", "vllm",

    # Good dtype for Blackwell.
    "--dtype", "bfloat16",

    # Safe for custom model code.
    "--trust-remote-code",
]

server_env = os.environ.copy()

# Avoid FlashInfer sampler crash in this environment.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# CUDA 13.0 / Blackwell hints.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 12288 --gpu-memory-utilization 0.92 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 8 --max-num-batched-tokens 16384 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 2647
Log: /content/vllm_server.log


In [6]:
#cell 6
# Wait for vLLM server.
import time
import requests
from pathlib import Path

def tail_log(path, n=80):
    # Return recent log lines.
    path = Path(path)

    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()

    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        health = requests.get(f"http://localhost:{PORT}/health", timeout=5)

        if health.status_code == 200:
            models = requests.get(f"{BASE_URL}/models", timeout=10)

            if models.status_code == 200:
                ready = True
                model_info = models.json()["data"][0]

                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break

    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")

        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)

        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=3193) INFO 06-02 13:27:55 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=3193) INFO 06-02 13:27:55 [parallel_state.py:1422] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:35667 backend=nccl
(EngineCore pid=3193) INFO 06-02 13:27:55 [parallel_state.py:1735] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=3193) INFO 06-02 13:27:55 [topk_topp_sampler.py:70] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0; using PyTorch-native sampler.
(EngineCore pid=3193) INFO 06-02 13:27:55 [gpu_model_runner.py:5089] Starting to load model Qwen/Qwen3.5-27B...
(EngineCore pid=3193) INFO 06-02 13:27:56 [cuda.py:433] Using backend AttentionBackendEnum.FLASH_ATTN for 

In [7]:
#cell 7
# Define the judge prompt and JSON schema.
JUDGE_SYSTEM_PROMPT = """
You are a strict but fair answer judge.

You will receive:
- a question
- the ground-truth answer
- a candidate LLM response

Your job is to judge whether the candidate response correctly answers the question.
Use the ground-truth answer as the authoritative answer.
Do not solve the question yourself.
Do not add background knowledge.
Do not write analysis outside JSON.

Return only this JSON object, with reason first and score second:
{
  "reason": "short reason",
  "score": 0 or 1
}

Rules:
- Score 1 if the candidate gives the correct answer to the question.
- Score 1 if the candidate is semantically the same as the ground truth, even with different wording.
- Score 1 if the candidate gives the main requested answer clearly and any missing part is only a harmless clarifying suffix.
- Score 1 if the candidate includes extra information that does not contradict the correct answer.
- Score 0 if the candidate is wrong.
- Score 0 if the candidate is incomplete for what the question asks.
- Score 0 if the candidate is vague, too broad, or too narrow.
- Score 0 if the candidate says the answer is unknown, unavailable, cannot be determined, or similar.
- Score 0 if the candidate contains a contradiction or an incorrect final answer.
- Use the question to decide what details are required.
- Do not require exact string matching.
- When unsure, score 0.

Keep the reason short: maximum 25 words.
""".strip()

JUDGE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "reason": {
            "type": "string"
        },
        "score": {
            "type": "integer",
            "enum": [0, 1]
        },
    },
    "required": ["reason", "score"],
    "additionalProperties": False,
}

def build_judge_user_prompt(question: str, gt: str, response: str) -> str:
    # Build one judging prompt.
    return f"""
Question:
{question}

Ground-truth answer:
{gt}

Candidate LLM response:
{response}

Judge the candidate response.
Return only JSON with reason first and score second.
""".strip()

In [8]:
#cell 8
# Create OpenAI-compatible client and JSON parser.
from openai import OpenAI
from typing import Any, Dict
import json
import time

client = OpenAI(
    base_url=BASE_URL,
    api_key="EMPTY",
)

def extract_first_json_object(text: str) -> Dict[str, Any]:
    # Extract the first JSON object.
    text = (text or "").strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")

    if start == -1:
        raise ValueError(f"No JSON object found: {text[:300]}")

    depth = 0
    in_str = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1

                if depth == 0:
                    return json.loads(text[start:i + 1])

    raise ValueError(f"Incomplete JSON object: {text[:300]}")

def normalize_judge_output(obj: Dict[str, Any]) -> Dict[str, Any]:
    # Normalize judge output.
    score = obj.get("score")

    if isinstance(score, str):
        score = score.strip()
        if score in {"0", "1"}:
            score = int(score)

    if score not in {0, 1}:
        raise ValueError(f"Invalid score: {obj}")

    reason = str(obj.get("reason", "")).strip()

    if not reason:
        reason = "No reason provided."

    # Keep reason compact.
    reason = " ".join(reason.split())

    return {
        "reason": reason,
        "score": int(score),
    }

In [9]:
#cell 9
# Define one judge call with guided JSON and retries.
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 512
MAX_RETRIES = 5

def judge_one_record(record: Dict[str, Any]) -> Dict[str, Any]:
    # Judge one row.
    question = str(record.get("question", ""))
    gt = str(record.get("gt", ""))
    response = str(record.get("response", ""))

    messages = [
        {
            "role": "system",
            "content": JUDGE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_judge_user_prompt(question, gt, response),
        },
    ]

    last_error = None
    started = time.perf_counter()

    for attempt in range(MAX_RETRIES):
        try:
            # First choice: vLLM guided JSON.
            try:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    temperature=JUDGE_TEMPERATURE,
                    max_tokens=JUDGE_MAX_TOKENS,
                    extra_body={
                        "guided_json": JUDGE_JSON_SCHEMA,
                    },
                )

            except Exception:
                # Fallback: OpenAI-style JSON mode.
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    temperature=JUDGE_TEMPERATURE,
                    max_tokens=JUDGE_MAX_TOKENS,
                    response_format={"type": "json_object"},
                )

            raw_text = completion.choices[0].message.content or ""
            parsed = normalize_judge_output(extract_first_json_object(raw_text))

            latency = time.perf_counter() - started

            return {
                "judge_reason": parsed["reason"],
                "judge_score": parsed["score"],
                "judge_raw": raw_text,
                "judge_error": None,
                "judge_latency_sec": latency,
            }

        except Exception as e:
            last_error = repr(e)
            time.sleep(2.0 * (attempt + 1))

    latency = time.perf_counter() - started

    return {
        "judge_reason": None,
        "judge_score": None,
        "judge_raw": None,
        "judge_error": last_error,
        "judge_latency_sec": latency,
    }

In [10]:
#cell 10
# Judge one file with resume support and automatic rejudging of failed rows.
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict, Any
import json

REQUEST_CONCURRENCY = 16
FILE_MAX_ROUNDS = 5
RAISE_ON_FAILED_JUDGEMENTS = True

def make_result_key(record: Dict[str, Any], row_id: int) -> str:
    # Make a stable key.
    if record.get("source_index") is not None:
        return str(record.get("source_index"))

    return str(row_id)

def is_valid_judgement(obj: Dict[str, Any] | None) -> bool:
    # Check if a saved judgement is usable.
    if obj is None:
        return False

    if obj.get("judge_error") is not None:
        return False

    if obj.get("judge_score") not in {0, 1}:
        return False

    return True

def load_existing_jsonl(path: Path) -> Dict[str, Dict[str, Any]]:
    # Load latest result for each key.
    existing = {}

    if not path.exists():
        return existing

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            key = str(obj.get("_judge_key"))
            existing[key] = obj

    return existing

def append_jsonl(path: Path, obj: Dict[str, Any]):
    # Append one JSONL row.
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def judge_file(input_file_name: str) -> Path:
    # Judge all rows in one file.
    data = all_input_data[input_file_name]

    stem = Path(input_file_name).stem
    out_jsonl = GDRIVE_OUTPUT_DIR / f"{stem}__judged.jsonl"
    out_json = GDRIVE_OUTPUT_DIR / f"{stem}__judged.json"

    for round_id in range(1, FILE_MAX_ROUNDS + 1):
        existing = load_existing_jsonl(out_jsonl)

        todo = []

        for row_id, record in enumerate(data):
            key = make_result_key(record, row_id)
            current = existing.get(key)

            # Rejudge missing or failed rows.
            if not is_valid_judgement(current):
                todo.append((row_id, key, record))

        print("\nFile:", input_file_name)
        print("Round:", round_id)
        print("Rows:", len(data))
        print("Valid already judged:", len(data) - len(todo))
        print("Remaining or failed:", len(todo))

        if not todo:
            break

        with ThreadPoolExecutor(max_workers=REQUEST_CONCURRENCY) as executor:
            futures = {
                executor.submit(judge_one_record, record): (row_id, key, record)
                for row_id, key, record in todo
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"Judging {input_file_name} | round {round_id}",
            ):
                row_id, key, record = futures[future]
                judge_result = future.result()

                out_obj = {
                    "_judge_key": key,
                    "_row_id": row_id,
                    "_input_file": input_file_name,

                    "source_index": record.get("source_index"),
                    "type": record.get("type"),
                    "question": record.get("question"),
                    "gt": record.get("gt"),
                    "response": record.get("response"),

                    "attempt": record.get("attempt"),
                    "max_tokens_used": record.get("max_tokens_used"),
                    "completion_tokens": record.get("completion_tokens"),
                    "retry_reason": record.get("retry_reason"),

                    **judge_result,
                }

                append_jsonl(out_jsonl, out_obj)

    # Reload after all rounds.
    existing = load_existing_jsonl(out_jsonl)

    ordered = []

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)

        if key not in existing:
            raise RuntimeError(f"Missing judgement for key={key} in {input_file_name}")

        ordered.append(existing[key])

    failed = [
        x for x in ordered
        if not is_valid_judgement(x)
    ]

    print("Final failed judgements:", len(failed))

    if failed and RAISE_ON_FAILED_JUDGEMENTS:
        print("First failed judgement:")
        print(json.dumps(failed[0], ensure_ascii=False, indent=2)[:2000])
        raise RuntimeError(
            f"{len(failed)} failed judgements remain in {input_file_name}. "
            f"Re-run this cell or reduce REQUEST_CONCURRENCY."
        )

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)

    print("Saved JSONL:", out_jsonl)
    print("Saved JSON:", out_json)

    return out_json

In [11]:
#cell 11
# Run judge for all files.
judged_json_paths = []

for input_file_name in INPUT_FILES:
    judged_path = judge_file(input_file_name)
    judged_json_paths.append(judged_path)

print("\nAll files judged.")

for path in judged_json_paths:
    print(path)


File: hotpotqa_answer_qwen3.5.json
Round: 1
Rows: 1000
Valid already judged: 997
Remaining or failed: 3


Judging hotpotqa_answer_qwen3.5.json | round 1:   0%|          | 0/3 [00:00<?, ?it/s]


File: hotpotqa_answer_qwen3.5.json
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_qwen3.5__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_qwen3.5__judged.json

File: hotpotqa_answer_gpt_oss.json
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging hotpotqa_answer_gpt_oss.json | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


File: hotpotqa_answer_gpt_oss.json
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_gpt_oss__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_gpt_oss__judged.json

File: hotpotqa_answer_gemma4.json
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


...

Judging hotpotqa_answer_gemma4.json | round 4:   0%|          | 0/1 [00:00<?, ?it/s]


File: hotpotqa_answer_gemma4.json
Round: 5
Rows: 1000
Valid already judged: 999
Remaining or failed: 1


Judging hotpotqa_answer_gemma4.json | round 5:   0%|          | 0/1 [00:00<?, ?it/s]

Final failed judgements: 1
First failed judgement:
{
  "_judge_key": "330",
  "_row_id": 330,
  "_input_file": "hotpotqa_answer_gemma4.json",
  "source_index": 330,
  "type": "bridge",
  "question": "What are the names of the members of the Detroit-based hip hop duo who has worked with Jason Gilbert?",
  "gt": "Royce da 5'9\" (Bad) and Eminem (Evil)",
  "response": "Royce da 5'9' and Eminem",
  "attempt": 1,
  "max_tokens_used": 65536,
  "completion_tokens": 419,
  "retry_reason": null,
  "judge_reason": null,
  "judge_score": null,
  "judge_raw": null,
  "judge_error": "ValueError('Incomplete JSON object: {\\n  \"reason\": \"The candidate correctly identifies the duo members Royce da 5\\'9\" and Eminem, matching the ground truth despite minor formatting differences.\",\\n  \"score\": 1\\n}')",
  "judge_latency_sec": 38.834742045999974
}


RuntimeError: 1 failed judgements remain in hotpotqa_answer_gemma4.json. Re-run this cell or reduce REQUEST_CONCURRENCY.

In [12]:
#cell 11.5
# Repair failed judgements before loading final results.
import re
import json
import time
from pathlib import Path
from tqdm.auto import tqdm

REPAIR_MAX_RETRIES = 3
REPAIR_MAX_TOKENS = 128
REPAIR_TEMPERATURE = 0.0

REPAIR_SYSTEM_PROMPT = """
You are a strict but fair answer judge.

You will receive:
- a question
- the ground-truth answer
- a candidate LLM response

Judge whether the candidate response correctly answers the question.
Use the ground-truth answer as the authoritative answer.
Do not solve the question yourself.

Return exactly two lines:
reason: short reason without quotation marks
score: 0 or 1

Rules:
- Score 1 if the candidate gives the correct answer.
- Score 1 if the candidate is semantically the same as the ground truth.
- Score 1 if extra information is present but not contradictory.
- Score 1 if the candidate gives the main requested answer clearly and any missing part is only a harmless clarifying suffix.
- Score 0 if the candidate is wrong, incomplete, vague, too broad, or too narrow.
- Score 0 if the candidate says unknown, unavailable, cannot be determined, or similar.
- Score 0 if there is contradiction or an incorrect final answer.
- When unsure, score 0.
""".strip()

def build_repair_prompt(question: str, gt: str, response: str) -> str:
    # Build repair judge prompt.
    return f"""
Question:
{question}

Ground-truth answer:
{gt}

Candidate LLM response:
{response}

Return exactly:
reason: ...
score: 0 or 1
""".strip()

def parse_repair_output(text: str):
    # Parse score and reason from plain text.
    text = (text or "").strip()

    score_match = re.search(r"(?im)^\s*score\s*:\s*([01])\s*$", text)

    if score_match is None:
        score_match = re.search(r"(?i)\bscore\s*[:=]\s*([01])\b", text)

    if score_match is None:
        raise ValueError(f"Could not parse score from: {text[:300]}")

    score = int(score_match.group(1))

    reason_match = re.search(r"(?im)^\s*reason\s*:\s*(.+?)\s*$", text)
    reason = reason_match.group(1).strip() if reason_match else "Parsed score from repair judge."

    reason = reason.replace('"', "'")
    reason = " ".join(reason.split())

    if not reason:
        reason = "No reason provided."

    return {
        "judge_reason": reason,
        "judge_score": score,
        "judge_raw": text,
        "judge_error": None,
    }

def repair_judge_one_record(record):
    # Rejudge one failed row with plain-text output.
    question = str(record.get("question", ""))
    gt = str(record.get("gt", ""))
    response = str(record.get("response", ""))

    messages = [
        {
            "role": "system",
            "content": REPAIR_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_repair_prompt(question, gt, response),
        },
    ]

    last_error = None
    started = time.perf_counter()

    for attempt in range(REPAIR_MAX_RETRIES):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=REPAIR_TEMPERATURE,
                max_tokens=REPAIR_MAX_TOKENS,
            )

            raw_text = completion.choices[0].message.content or ""
            parsed = parse_repair_output(raw_text)

            parsed["judge_latency_sec"] = time.perf_counter() - started
            return parsed

        except Exception as e:
            last_error = repr(e)
            time.sleep(1.5 * (attempt + 1))

    # Final fallback requested by user.
    return {
        "judge_reason": "Repair failed after retries; defaulted to incorrect.",
        "judge_score": 0,
        "judge_raw": None,
        "judge_error": None,
        "judge_latency_sec": time.perf_counter() - started,
    }

def load_latest_jsonl(path: Path):
    # Load latest row for each key.
    latest = {}

    if not path.exists():
        return latest

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            latest[str(obj.get("_judge_key"))] = obj

    return latest

def valid_judgement(obj):
    # Check valid judgement.
    if obj is None:
        return False

    if obj.get("judge_error") is not None:
        return False

    if obj.get("judge_score") not in {0, 1}:
        return False

    return True

def append_jsonl_row(path: Path, obj):
    # Append repaired row.
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

judged_json_paths = []

for input_file_name in INPUT_FILES:
    data = all_input_data[input_file_name]

    stem = Path(input_file_name).stem
    out_jsonl = GDRIVE_OUTPUT_DIR / f"{stem}__judged.jsonl"
    out_json = GDRIVE_OUTPUT_DIR / f"{stem}__judged.json"

    latest = load_latest_jsonl(out_jsonl)

    failed_items = []

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)
        current = latest.get(key)

        if not valid_judgement(current):
            failed_items.append((row_id, key, record, current))

    print("\nFile:", input_file_name)
    print("Failed or missing before repair:", len(failed_items))

    for row_id, key, record, old_obj in tqdm(failed_items, desc=f"Repairing {input_file_name}"):
        repair_result = repair_judge_one_record(record)

        repaired_obj = {
            "_judge_key": key,
            "_row_id": row_id,
            "_input_file": input_file_name,

            "source_index": record.get("source_index"),
            "type": record.get("type"),
            "question": record.get("question"),
            "gt": record.get("gt"),
            "response": record.get("response"),

            "attempt": record.get("attempt"),
            "max_tokens_used": record.get("max_tokens_used"),
            "completion_tokens": record.get("completion_tokens"),
            "retry_reason": record.get("retry_reason"),

            **repair_result,
        }

        append_jsonl_row(out_jsonl, repaired_obj)
        latest[key] = repaired_obj

    # Build ordered final JSON.
    ordered = []

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)
        obj = latest.get(key)

        if not valid_judgement(obj):
            # Last safety fallback.
            obj = {
                "_judge_key": key,
                "_row_id": row_id,
                "_input_file": input_file_name,

                "source_index": record.get("source_index"),
                "type": record.get("type"),
                "question": record.get("question"),
                "gt": record.get("gt"),
                "response": record.get("response"),

                "attempt": record.get("attempt"),
                "max_tokens_used": record.get("max_tokens_used"),
                "completion_tokens": record.get("completion_tokens"),
                "retry_reason": record.get("retry_reason"),

                "judge_reason": "Final fallback after repair failure; defaulted to incorrect.",
                "judge_score": 0,
                "judge_raw": None,
                "judge_error": None,
                "judge_latency_sec": None,
            }

            append_jsonl_row(out_jsonl, obj)
            latest[key] = obj

        ordered.append(obj)

    final_failed = [x for x in ordered if not valid_judgement(x)]

    print("Failed after repair:", len(final_failed))

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)

    print("Saved final JSON:", out_json)

    judged_json_paths.append(out_json)

print("\nRepair complete. Now run cell 12.")
for path in judged_json_paths:
    print(path)


File: hotpotqa_answer_qwen3.5.json
Failed or missing before repair: 0


Repairing hotpotqa_answer_qwen3.5.json: 0it [00:00, ?it/s]

Failed after repair: 0
Saved final JSON: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_qwen3.5__judged.json

File: hotpotqa_answer_gpt_oss.json
Failed or missing before repair: 0


Repairing hotpotqa_answer_gpt_oss.json: 0it [00:00, ?it/s]

Failed after repair: 0
Saved final JSON: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_gpt_oss__judged.json

File: hotpotqa_answer_gemma4.json
Failed or missing before repair: 1


Repairing hotpotqa_answer_gemma4.json:   0%|          | 0/1 [00:00<?, ?it/s]

Failed after repair: 0
Saved final JSON: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_gemma4__judged.json

Repair complete. Now run cell 12.
/content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_qwen3.5__judged.json
/content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_gpt_oss__judged.json
/content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/llm_judge_qwen35_27b/hotpotqa_answer_gemma4__judged.json


In [13]:
#cell 12
# Load all judged outputs.
import pandas as pd
import json

rows = []

for path in judged_json_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows.extend(data)

df = pd.DataFrame(rows)

df["judge_score"] = pd.to_numeric(df["judge_score"], errors="coerce")

print("Total rows:", len(df))
print("Files:", df["_input_file"].nunique())
print("Invalid judgements:", df["judge_score"].isna().sum())

display(df.head())

Total rows: 3000
Files: 3
Invalid judgements: 0


,_judge_key,_row_id,_input_file,source_index,type,question,gt,response,attempt,max_tokens_used,completion_tokens,retry_reason,judge_score,judge_reason,judge_raw,judge_error,judge_latency_sec
0,0,0,hotpotqa_answer_qwen3.5.json,0,bridge,What government position was held by the woman...,Chief of Protocol,United States Ambassador and Chief of Protocol,0,32768,4175,None,1,The candidate correctly identifies the positio...,"{\n ""score"": 1,\n ""reason"": ""The candidate c...",None,25.853335
1,1,1,hotpotqa_answer_qwen3.5.json,1,bridge,"What science fantasy young adult series, told ...",Animorphs,Animorphs,0,32768,1031,None,1,The candidate response matches the ground trut...,"{\n ""score"": 1,\n ""reason"": ""The candidate r...",None,23.980586
2,2,2,hotpotqa_answer_qwen3.5.json,2,bridge,"The director of the romantic comedy ""Big Stone...","Greenwich Village, New York City",New York City,0,32768,1584,None,0,The question asks for a specific city within N...,"{\n ""score"": 0,\n ""reason"": ""The question as...",None,25.407243
3,3,3,hotpotqa_answer_qwen3.5.json,3,bridge,2014 S/S is the debut album of a South Korean ...,YG Entertainment,YG Entertainment,0,32768,1606,None,1,The candidate response correctly identifies YG...,"{\n ""score"": 1,\n ""reason"": ""The candidate r...",None,25.204830
4,4,4,hotpotqa_answer_qwen3.5.json,4,bridge,Who was known by his stage name Aladin and hel...,Eenasul Fateh,Eenasul Fateh,0,32768,918,None,1,The candidate response matches the ground trut...,"{\n ""score"": 1,\n ""reason"": ""The candidate r...",None,23.980621


In [14]:
#cell 13
# Compute accuracy overall and by type.
valid_df = df[df["judge_score"].isin([0, 1])].copy()

if len(valid_df) != len(df):
    raise RuntimeError("Some judgements are invalid. Fix them before computing accuracy.")

def accuracy_table(dataframe, group_cols):
    # Build accuracy table.
    result = (
        dataframe
        .groupby(group_cols, dropna=False)
        .agg(
            total=("judge_score", "size"),
            correct=("judge_score", "sum"),
            accuracy=("judge_score", "mean"),
        )
        .reset_index()
    )

    result["correct"] = result["correct"].astype(int)
    result["accuracy_percent"] = result["accuracy"] * 100.0

    return result.sort_values(group_cols).reset_index(drop=True)

per_file_overall = accuracy_table(valid_df, ["_input_file"])
per_file_type = accuracy_table(valid_df, ["_input_file", "type"])
combined_type = accuracy_table(valid_df, ["type"])

combined_overall = pd.DataFrame([
    {
        "scope": "all_files_combined",
        "total": int(valid_df["judge_score"].size),
        "correct": int(valid_df["judge_score"].sum()),
        "accuracy": float(valid_df["judge_score"].mean()),
        "accuracy_percent": float(valid_df["judge_score"].mean() * 100.0),
    }
])

print("Per-file overall accuracy:")
display(per_file_overall)

print("Per-file per-type accuracy:")
display(per_file_type)

print("Combined per-type accuracy:")
display(combined_type)

print("Combined overall accuracy:")
display(combined_overall)

Per-file overall accuracy:


,_input_file,total,correct,accuracy,accuracy_percent
0,hotpotqa_answer_gemma4.json,1000,867,0.867,86.7
1,hotpotqa_answer_gpt_oss.json,1000,861,0.861,86.1
2,hotpotqa_answer_qwen3.5.json,1000,895,0.895,89.5


Per-file per-type accuracy:


,_input_file,type,total,correct,accuracy,accuracy_percent
0,hotpotqa_answer_gemma4.json,bridge,700,594,0.848571,84.857143
1,hotpotqa_answer_gemma4.json,comparison,300,273,0.910000,91.000000
2,hotpotqa_answer_gpt_oss.json,bridge,700,586,0.837143,83.714286
3,hotpotqa_answer_gpt_oss.json,comparison,300,275,0.916667,91.666667
4,hotpotqa_answer_qwen3.5.json,bridge,700,616,0.880000,88.000000
5,hotpotqa_answer_qwen3.5.json,comparison,300,279,0.930000,93.000000


Combined per-type accuracy:


,type,total,correct,accuracy,accuracy_percent
0,bridge,2100,1796,0.855238,85.523810
1,comparison,900,827,0.918889,91.888889


Combined overall accuracy:


,scope,total,correct,accuracy,accuracy_percent
0,all_files_combined,3000,2623,0.874333,87.433333


In [15]:
#cell 15
# Inspect rejected answers.
N_EXAMPLES = 30

rejected = valid_df[valid_df["judge_score"] == 0].copy()

cols = [
    "_input_file",
    "source_index",
    "type",
    "question",
    "gt",
    "response",
    "judge_reason",
]

print("Rejected rows:", len(rejected))
display(rejected[cols].head(N_EXAMPLES))

Rejected rows: 377


,_input_file,source_index,type,question,gt,response,judge_reason
2,hotpotqa_answer_qwen3.5.json,2,bridge,"The director of the romantic comedy ""Big Stone...","Greenwich Village, New York City",New York City,The question asks for a specific city within N...
11,hotpotqa_answer_qwen3.5.json,11,bridge,Brown State Fishing Lake is in a country that ...,"9,984",Information not available,The candidate response states that the informa...
22,hotpotqa_answer_qwen3.5.json,22,bridge,"When was Poison's album ""Shut Up, Make Love"" r...",2000,Information not available,The candidate response states that the informa...
24,hotpotqa_answer_qwen3.5.json,24,bridge,Alexander Kerensky was defeated and destroyed ...,October 1922,1923,The candidate answer '1923' is incorrect. The ...
35,hotpotqa_answer_qwen3.5.json,35,bridge,Which British first-generation jet-powered med...,English Electric Canberra,Information not available,The candidate response states that the informa...
41,hotpotqa_answer_qwen3.5.json,41,bridge,Which Australian city founded in 1838 contains...,"Marion, South Australia",Adelaide,The question asks for a specific city founded ...
61,hotpotqa_answer_qwen3.5.json,61,bridge,Alvaro Mexia had a diplomatic mission with whi...,Apalachees,The Ais,The candidate response 'The Ais' is incorrect....
71,hotpotqa_answer_qwen3.5.json,71,bridge,How many copies of Roald Dahl's variation on a...,250 million,Information not available,The candidate response states that the informa...
79,hotpotqa_answer_qwen3.5.json,79,bridge,Where did the descendants of the group of blac...,"Coahuila, Mexico","Oklahoma, Florida, Texas, the Bahamas, and nor...",The candidate response lists multiple location...
81,hotpotqa_answer_qwen3.5.json,81,bridge,What year was the brother of this first round ...,2003,2006,The candidate response (2006) does not match t...


In [16]:
#cell 16
# Inspect accepted answers.
N_EXAMPLES = 30

accepted = valid_df[valid_df["judge_score"] == 1].copy()

cols = [
    "_input_file",
    "source_index",
    "type",
    "question",
    "gt",
    "response",
    "judge_reason",
]

print("Accepted rows:", len(accepted))
display(accepted[cols].head(N_EXAMPLES))

Accepted rows: 2623


,_input_file,source_index,type,question,gt,response,judge_reason
0,hotpotqa_answer_qwen3.5.json,0,bridge,What government position was held by the woman...,Chief of Protocol,United States Ambassador and Chief of Protocol,The candidate correctly identifies the positio...
1,hotpotqa_answer_qwen3.5.json,1,bridge,"What science fantasy young adult series, told ...",Animorphs,Animorphs,The candidate response matches the ground trut...
3,hotpotqa_answer_qwen3.5.json,3,bridge,2014 S/S is the debut album of a South Korean ...,YG Entertainment,YG Entertainment,The candidate response correctly identifies YG...
4,hotpotqa_answer_qwen3.5.json,4,bridge,Who was known by his stage name Aladin and hel...,Eenasul Fateh,Eenasul Fateh,The candidate response matches the ground trut...
5,hotpotqa_answer_qwen3.5.json,5,bridge,The arena where the Lewiston Maineiacs played ...,"3,677 seated","3,677",The candidate response provides the exact corr...
6,hotpotqa_answer_qwen3.5.json,6,bridge,"Who is older, Annie Morton or Terry Richardson?",Terry Richardson,Terry Richardson,The candidate response matches the ground trut...
7,hotpotqa_answer_qwen3.5.json,7,bridge,What is the name of the fight song of the univ...,Kansas Song,Kansas Song,The candidate response correctly identifies 'K...
8,hotpotqa_answer_qwen3.5.json,8,bridge,"What screenwriter with credits for ""Evolution""...",David Weissman,David Weissman,The candidate response correctly identifies Da...
9,hotpotqa_answer_qwen3.5.json,9,bridge,What year did Guns N Roses perform a promo for...,1999,1999,The candidate response matches the ground trut...
10,hotpotqa_answer_qwen3.5.json,10,bridge,The football manager who recruited David Beckh...,from 1986 to 2013,1986 to 2013,The candidate response matches the ground trut...
